In [ ]:
# -------- CELL --------
# 🏆 KAGGLE PLAYGROUND S6E7 GRANDMASTER V3 MASTER PIPELINE (LIGHTGBM OPENCL BUG FIXED)
# Strategy: Pure v3 Proven Baseline + Raw OOF Target Encoding (3 Combos) + Integer Frequencies + CatBoost Triad + Two-Stage Scipy Optimization (Model Weights + Class Multipliers)

import os
import gc
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder
from scipy.optimize import minimize
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')
print("Environment initialized successfully for Grandmaster v3 Master Pipeline.")


Environment initialized successfully for Grandmaster v3 Master Pipeline.


In [ ]:
# -------- CELL --------
# 1. Load Data
DATA_PATH = './data'
if not os.path.exists(DATA_PATH):
    DATA_PATH = 'c:/Users/mihisara/Desktop/ML/data'

train = pd.read_csv(f'{DATA_PATH}/train.csv')
test = pd.read_csv(f'{DATA_PATH}/test.csv')
sub_file = f'{DATA_PATH}/sample_submission.csv'
if os.path.exists(sub_file):
    sample_submission = pd.read_csv(sub_file)
else:
    sample_submission = pd.DataFrame({'id': test['id'], 'health_condition': 'at-risk'})

TARGET = 'health_condition'
if TARGET not in train.columns:
    TARGET = [c for c in train.columns if c not in test.columns][0]
ID_COLUMN = sample_submission.columns[0]

drop_columns = [TARGET]
if ID_COLUMN in train.columns: drop_columns.append(ID_COLUMN)

X = train.drop(columns=drop_columns).copy()
y_text = train[TARGET].copy()
test_features = test.drop(columns=[ID_COLUMN], errors='ignore').copy()
print(f'Train shape: {X.shape}, Test shape: {test_features.shape}')


Train shape: (690088, 13), Test shape: (295753, 13)


In [ ]:
# -------- CELL --------
# 2. 100% Proven v3 Master Feature Engineering (Train-Only Statistics & Integer Frequency)
def add_v3_master_features(df_tr, df_te):
    df_tr = df_tr.copy()
    df_te = df_te.copy()

    # 1. Missingness-Aware Flags
    orig_cols = df_tr.columns.tolist()
    df_tr['row_missing_count'] = df_tr[orig_cols].isnull().sum(axis=1).astype('int8')
    df_te['row_missing_count'] = df_te[orig_cols].isnull().sum(axis=1).astype('int8')
    for col in orig_cols:
        df_tr[f'{col}__missing'] = df_tr[col].isna().astype('int8')
        df_te[f'{col}__missing'] = df_te[col].isna().astype('int8')

    lower_map = {c.lower(): c for c in df_tr.columns}
    def find_col(possible_names):
        for name in possible_names:
            if name in lower_map: return lower_map[name]
        return None

    sleep_col = find_col(['sleep_duration', 'sleep_hours', 'sleep'])
    exercise_col = find_col(['exercise_duration'])
    stress_col = find_col(['stress_level'])
    activity_col = find_col(['physical_activity_level', 'physical_activity'])
    quality_col = find_col(['sleep_quality'])
    smoke_col = find_col(['smoking_alcohol'])
    step_col = find_col(['step_count'])
    water_col = find_col(['water_intake'])
    hr_col = find_col(['heart_rate'])
    bmi_col = find_col(['bmi'])
    cal_col = find_col(['calorie_expenditure'])
    gender_col = find_col(['gender'])

    # 2. Medical & Domain Bins
    for df in [df_tr, df_te]:
        if bmi_col:
            bv = df[bmi_col]
            bcat = np.zeros(len(df), dtype='int8')
            bcat[bv < 18.5] = 1
            bcat[(bv >= 18.5) & (bv < 25.0)] = 2
            bcat[(bv >= 25.0) & (bv < 30.0)] = 3
            bcat[bv >= 30.0] = 4
            df['bmi_category'] = bcat

        if hr_col:
            hv = df[hr_col]
            hz = np.zeros(len(df), dtype='int8')
            hz[hv < 60.0] = 1
            hz[(hv >= 60.0) & (hv <= 100.0)] = 2
            hz[hv > 100.0] = 3
            df['hr_zone'] = hz

        # 3. Lifestyle Composite Risk Index
        risk = np.zeros(len(df), dtype='int8')
        if sleep_col: risk += (df[sleep_col] < 6.0).astype('int8')
        if stress_col: risk += (df[stress_col].astype(str) == 'high').astype('int8')
        if quality_col: risk += (df[quality_col].astype(str) == 'poor').astype('int8')
        if activity_col: risk += (df[activity_col].astype(str) == 'sedentary').astype('int8')
        if smoke_col: risk += (df[smoke_col].astype(str) == 'yes').astype('int8')
        if bmi_col: risk += (df[bmi_col] >= 25.0).astype('int8')
        df['lifestyle_risk_index'] = risk

        # 4. Domain Ratios & Interactions (Proven v3)
        if sleep_col:
            df['sleep_distance_from_8'] = (df[sleep_col] - 8.0).abs()
            df['sleep_ge_7'] = (df[sleep_col] >= 7.0).astype(int)
            df['sleep_lt_6'] = (df[sleep_col] < 6.0).astype(int)

        if cal_col and step_col: df['calories_per_step'] = df[cal_col] / (df[step_col] + 1.0)
        if cal_col and exercise_col: df['calories_per_exercise_min'] = df[cal_col] / (df[exercise_col] + 1.0)
        if cal_col and bmi_col: df['calories_per_bmi'] = df[cal_col] / (df[bmi_col] + 0.01)
        if water_col and bmi_col: df['water_per_bmi'] = df[water_col] / (df[bmi_col] + 0.01)
        if step_col and bmi_col: df['step_per_bmi'] = df[step_col] / (df[bmi_col] + 0.01)
        if hr_col and sleep_col: df['hr_per_sleep'] = df[hr_col] / (df[sleep_col] + 0.01)
        if hr_col and exercise_col: df['hr_per_exercise'] = df[hr_col] / (df[exercise_col] + 1.0)
        if sleep_col and exercise_col:
            df['sleep_exercise_interaction'] = df[sleep_col] * df[exercise_col]
            df['sleep_exercise_ratio'] = df[sleep_col] / (df[exercise_col] + 0.01)
        if water_col and step_col: df['water_per_step'] = df[water_col] / (df[step_col] + 1.0)
        if exercise_col and step_col: df['exercise_per_step'] = df[exercise_col] / (df[step_col] + 1.0)

        # Categorical Combinations
        if stress_col and activity_col:
            df['stress_activity_combo'] = df[stress_col].astype(str) + "_" + df[activity_col].astype(str)
        if quality_col and stress_col:
            df['sleep_stress_combo'] = df[quality_col].astype(str) + "_" + df[stress_col].astype(str)
        if stress_col and activity_col and quality_col:
            df['lifestyle_triad'] = df[stress_col].astype(str) + "_" + df[activity_col].astype(str) + "_" + df[quality_col].astype(str)

    # 5. LEAK-FREE Peer Group Aggregations
    if sleep_col and activity_col:
        grp = df_tr.groupby(activity_col)[sleep_col].mean().to_dict()
        df_tr['sleep_diff_from_activity_mean'] = df_tr[sleep_col] - df_tr[activity_col].map(grp)
        df_te['sleep_diff_from_activity_mean'] = df_te[sleep_col] - df_te[activity_col].map(grp)

    if exercise_col and stress_col:
        grp = df_tr.groupby(stress_col)[exercise_col].mean().to_dict()
        df_tr['exercise_diff_from_stress_mean'] = df_tr[exercise_col] - df_tr[stress_col].map(grp)
        df_te['exercise_diff_from_stress_mean'] = df_te[exercise_col] - df_te[stress_col].map(grp)

    if cal_col and activity_col:
        grp = df_tr.groupby(activity_col)[cal_col].mean().to_dict()
        df_tr['calories_diff_from_activity_mean'] = df_tr[cal_col] - df_tr[activity_col].map(grp)
        df_te['calories_diff_from_activity_mean'] = df_te[cal_col] - df_te[activity_col].map(grp)

    # 6. LEAK-FREE Integer Frequency Encoding (Exact v3 Integer Count Mapping)
    cat_for_freq = [stress_col, activity_col, quality_col, smoke_col, gender_col, 'stress_activity_combo', 'sleep_stress_combo', 'lifestyle_triad']
    for c in cat_for_freq:
        if c and c in df_tr.columns:
            freq = df_tr[c].value_counts(dropna=False).to_dict()
            df_tr[f'{c}__freq'] = df_tr[c].map(freq).fillna(0).astype('int32')
            df_te[f'{c}__freq'] = df_te[c].map(freq).fillna(0).astype('int32')

    return df_tr, df_te

X, test_features = add_v3_master_features(X, test_features)
print(f'Train shape after Leak-Free Feature Engineering: {X.shape}')


Train shape after Leak-Free Feature Engineering: (690088, 58)


In [ ]:
# -------- CELL --------
# 3. Raw OOF Target Encoding (3 High-Cardinality Combos Only - Zero Noise)
le = LabelEncoder()
y = le.fit_transform(y_text)
num_classes = len(le.classes_)

te_cols = ['stress_activity_combo', 'sleep_stress_combo', 'lifestyle_triad']
skf_te = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Computing Raw OOF Target Encodings (Exact v3 Method)...")
for c in te_cols:
    if c in X.columns:
        for cls in range(num_classes):
            col_name = f'{c}__te_cls{cls}'
            X[col_name] = 0.0
            test_features[col_name] = 0.0

            y_cls = (y == cls).astype(float)
            for tr_idx, va_idx in skf_te.split(X, y):
                grp_means = X.iloc[tr_idx].groupby(c).apply(lambda d: y_cls[d.index].mean()).to_dict()
                global_mean = y_cls[tr_idx].mean()
                X.iloc[va_idx, X.columns.get_loc(col_name)] = X.iloc[va_idx][c].map(grp_means).fillna(global_mean)

            full_grp_means = X.groupby(c).apply(lambda d: y_cls[d.index].mean()).to_dict()
            global_mean = y_cls.mean()
            test_features[col_name] = test_features[c].map(full_grp_means).fillna(global_mean)

# Clean string formatting for LightGBM, XGBoost, and CatBoost
categorical_columns = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
for col in categorical_columns:
    X[col] = X[col].astype(str).replace(['nan', 'None', 'NaN', 'N/A'], 'missing').astype('category')
    test_features[col] = test_features[col].astype(str).replace(['nan', 'None', 'NaN', 'N/A'], 'missing').astype('category')

print(f'Final Train shape after Raw OOF Target Encoding: {X.shape}')


Computing Raw OOF Target Encodings (Exact v3 Method)...
Final Train shape after Raw OOF Target Encoding: (690088, 67)


In [ ]:
# -------- CELL --------
# 4. Multi-Seed 5-Fold Triad Ensemble (5 Seeds x 5 Folds = 75 Models Total)
SEEDS = [42, 2026, 999, 777, 3407]
oof_lgb = np.zeros((len(X), num_classes))
oof_xgb = np.zeros((len(X), num_classes))
oof_cat = np.zeros((len(X), num_classes))

test_lgb = np.zeros((len(test_features), num_classes))
test_xgb = np.zeros((len(test_features), num_classes))
test_cat = np.zeros((len(test_features), num_classes))

# LightGBM uses all CPU cores (n_jobs=-1) for lightning fast CPU execution without OpenCL dependency
lgb_params = {'n_estimators': 850, 'learning_rate': 0.03, 'num_leaves': 63, 'max_depth': 7, 'min_child_samples': 45, 'subsample': 0.8, 'colsample_bytree': 0.8, 'objective': 'multiclass', 'num_class': num_classes, 'n_jobs': -1, 'verbose': -1}
xgb_params = {'n_estimators': 850, 'learning_rate': 0.03, 'max_depth': 6, 'subsample': 0.8, 'colsample_bytree': 0.8, 'objective': 'multi:softprob', 'num_class': num_classes, 'eval_metric': 'mlogloss', 'enable_categorical': True, 'n_jobs': -1, 'tree_method': 'hist', 'device': 'cuda', 'early_stopping_rounds': 50}
cat_params = {'iterations': 850, 'learning_rate': 0.03, 'depth': 6, 'loss_function': 'MultiClass', 'task_type': 'GPU', 'verbose': 0}

cat_cols = X.select_dtypes(include=['category', 'object']).columns.tolist()
X_cat = X.copy()
test_features_cat = test_features.copy()
for col in cat_cols:
    X_cat[col] = X_cat[col].astype(str).replace(['nan', 'None', 'NaN', 'N/A'], 'missing')
    test_features_cat[col] = test_features_cat[col].astype(str).replace(['nan', 'None', 'NaN', 'N/A'], 'missing')

print(f'Starting 5-Seed 5-Fold Triad Ensemble ({len(SEEDS)*5*3} Models Total)...')
for seed in SEEDS:
    print(f'\n--- Training Seed {seed} ---')
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):
        X_tr, X_va = X.iloc[train_idx], X.iloc[valid_idx]
        y_tr, y_va = y[train_idx], y[valid_idx]
        X_tr_cat, X_va_cat = X_cat.iloc[train_idx], X_cat.iloc[valid_idx]

        # 1. LightGBM (Multi-threaded CPU)
        p = lgb_params.copy(); p['random_state'] = seed
        m1 = lgb.LGBMClassifier(**p); m1.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(50, verbose=False)])
        oof_lgb[valid_idx] += m1.predict_proba(X_va) / len(SEEDS)
        test_lgb += m1.predict_proba(test_features) / (5.0 * len(SEEDS))

        # 2. XGBoost (CUDA GPU)
        p = xgb_params.copy(); p['random_state'] = seed
        m2 = xgb.XGBClassifier(**p); m2.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        oof_xgb[valid_idx] += m2.predict_proba(X_va) / len(SEEDS)
        test_xgb += m2.predict_proba(test_features) / (5.0 * len(SEEDS))

        # 3. CatBoost (CUDA GPU)
        p = cat_params.copy(); p['random_seed'] = seed
        m3 = CatBoostClassifier(**p)
        m3.fit(X_tr_cat, y_tr, cat_features=cat_cols, eval_set=(X_va_cat, y_va), early_stopping_rounds=50, verbose=False)
        oof_cat[valid_idx] += m3.predict_proba(X_va_cat) / len(SEEDS)
        test_cat += m3.predict_proba(test_features_cat) / (5.0 * len(SEEDS))

print(f'\n5-Seed LightGBM Raw OOF Score: {balanced_accuracy_score(y, np.argmax(oof_lgb, axis=1)):.5f}')
print(f'5-Seed XGBoost Raw OOF Score:  {balanced_accuracy_score(y, np.argmax(oof_xgb, axis=1)):.5f}')
print(f'5-Seed CatBoost Raw OOF Score: {balanced_accuracy_score(y, np.argmax(oof_cat, axis=1)):.5f}')


Starting 5-Seed 5-Fold Triad Ensemble (75 Models Total)...

--- Training Seed 42 ---

--- Training Seed 2026 ---

--- Training Seed 999 ---

--- Training Seed 777 ---

--- Training Seed 3407 ---


In [ ]:
# -------- CELL --------
# 5. Two-Stage Scipy Optimization (Stage 1: Model Blend Weights + Stage 2: Class Weight Multipliers)
print('\nStage 1: Optimizing Model Blending Weights (LGBM, XGB, CatBoost)...')
def model_blend_objective(weights):
    w1, w2, w3 = weights
    blend = w1 * oof_lgb + w2 * oof_xgb + w3 * oof_cat
    return -balanced_accuracy_score(y, np.argmax(blend, axis=1))

res_m = minimize(model_blend_objective, [0.15, 0.15, 0.70], method='Nelder-Mead', bounds=[(0.0, 3.0)]*3)
w1, w2, w3 = res_m.x
print(f'Optimized Model Blending Weights (LGB, XGB, Cat): [{w1:.4f}, {w2:.4f}, {w3:.4f}]')

blended_oof = w1 * oof_lgb + w2 * oof_xgb + w3 * oof_cat
blended_test = w1 * test_lgb + w2 * test_xgb + w3 * test_cat

raw_blend_cv = balanced_accuracy_score(y, np.argmax(blended_oof, axis=1))
print(f'Raw Blended Triad OOF Balanced Accuracy: {raw_blend_cv:.5f}')

print('\nStage 2: Optimizing Class-Wise Multipliers on Blended OOF...')
def class_weight_objective(weights):
    adj = blended_oof * weights
    return -balanced_accuracy_score(y, np.argmax(adj, axis=1))

res_c = minimize(class_weight_objective, np.ones(num_classes), method='Nelder-Mead', bounds=[(0.1, 2.0)] * num_classes)
class_weights = res_c.x
print(f'Optimized Class Weights: {class_weights}')

final_oof_probs = blended_oof * class_weights
opt_preds = np.argmax(final_oof_probs, axis=1)
master_cv_score = balanced_accuracy_score(y, opt_preds)
print(f'\n🏆 FINAL GRANDMASTER V3 MASTER OOF BALANCED ACCURACY (TARGET: 0.95260+): {master_cv_score:.5f}')

print('\n--- Diagnostic Error Analysis ---')
cm = confusion_matrix(y, opt_preds)
print("Confusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y, opt_preds, target_names=le.classes_))

class_recalls = cm.diagonal() / cm.sum(axis=1)
for idx, class_name in enumerate(le.classes_):
    print(f"Recall for class '{class_name}': {class_recalls[idx]:.5f}")


In [ ]:
# -------- CELL --------
# 6. Final Predictions & Winner Production Submission Export
final_test_probs = blended_test * class_weights
final_preds = np.argmax(final_test_probs, axis=1)
final_labels = le.inverse_transform(final_preds)

submission = sample_submission.copy()
submission[TARGET] = final_labels
submission.to_csv('submission_grandmaster_v3_master_winner.csv', index=False)
print('Saved submission_grandmaster_v3_master_winner.csv successfully!')

submission.head()
